# AI Arena — PPO Curriculum Trainer (self-contained, 8h Kaggle GPU)

**Just paste this notebook into a fresh Kaggle notebook and click Run All.**

No external dataset uploads. Everything (env, GA-best opponent, curriculum,
behavior cloning, PPO, ONNX export) lives in this notebook.

Settings:
- **Accelerator**: GPU T4 ×2 (top-right of Kaggle notebook editor)
- **Internet**: ON (for the one-time `pip install`)

Output: `/kaggle/working/ppo_ckpt/model.onnx` — drop into the AshGrid repo
at `ai_arena/onnx/model.onnx`.

---

What this trains:
- 65-dim obs → 128 → 128 → 18-action discrete policy
- 4-stage curriculum (small/close → medium → near-deploy → deployment scale)
- Behavior cloning warm-start from GA-best (skips ~5M steps of "discover the fire button")
- Decaying reward shaping (visibility/approach/aim) → final policy uses pure kill/death
- 7.5h training budget by default — change `TIME_LIMIT_HOURS` to suit

## 1. Config — edit these then Run All

In [ ]:
# === TIME BUDGET ===
TIME_LIMIT_HOURS    = 7.5          # Kaggle session is 9h — leave 1.5h margin
                                    # for BC + ONNX export + sanity test

# === Smoke test (3 min) — flips to True for end-to-end verification ===
SMOKE_TEST          = False
if SMOKE_TEST:
    TIME_LIMIT_HOURS    = 0.05     # 3 min total
    BC_ROLLOUT_STEPS    = 20_000
    BC_EPOCHS           = 10
else:
    BC_ROLLOUT_STEPS    = 500_000  # GA-vs-GA rollout for behavior cloning (~5 min)
    BC_EPOCHS           = 50       # supervised epochs (~3 min on GPU)

# === PPO budget (ceiling — time limit will hit first in real training) ===
TOTAL_STEPS         = 64_000_000
N_ENVS              = 8

# === Self-play / curriculum bookkeeping ===
CHECKPOINT_EVERY    = 2_000_000    # save .zip every N real steps
FROZEN_OPP_EVERY    = 1_000_000    # snapshot policy into self-play pool
SELF_POOL_MAX       = 8

# === PPO hyperparams ===
LR_INIT             = 3e-4
ENT_COEF_INIT       = 0.02         # high exploration early
ENT_COEF_FINAL      = 0.005        # tighter exploitation late
N_EPOCHS            = 10
N_STEPS             = 2048
BATCH_SIZE          = 256
GAMMA               = 0.995
GAE_LAMBDA          = 0.95
CLIP_RANGE          = 0.2

# === Output ===
import os
OUTPUT_DIR = '/kaggle/working/ppo_ckpt'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'TIME_LIMIT_HOURS = {TIME_LIMIT_HOURS}h')
print(f'TOTAL_STEPS ceiling = {TOTAL_STEPS:,}')
print(f'BC pairs = {BC_ROLLOUT_STEPS:,}')
print(f'N_ENVS = {N_ENVS}, output = {OUTPUT_DIR}')

## 2. Install dependencies

In [ ]:
%pip install -q stable-baselines3 onnx onnxruntime

## 3. Materialize `combat_env.py` (the env, GA opponent, curriculum)

The full env source is embedded below as a base64 string. We write it to
`/kaggle/working/ai_arena_module/combat_env.py` and add that to `sys.path`,
so `import combat_env` works from worker processes too.

In [ ]:
import os, sys, base64

COMBAT_ENV_B64 = '''...'''

_module_dir = '/kaggle/working/ai_arena_module'
os.makedirs(_module_dir, exist_ok=True)
_env_path = os.path.join(_module_dir, 'combat_env.py')
with open(_env_path, 'wb') as f:
    f.write(base64.b64decode(COMBAT_ENV_B64))
print(f'Wrote {_env_path} ({os.path.getsize(_env_path):,} bytes)')

if _module_dir not in sys.path:
    sys.path.insert(0, _module_dir)

# Force a clean import (in case of re-runs)
for mod in list(sys.modules.keys()):
    if mod.startswith('combat_env'):
        del sys.modules[mod]

import combat_env
from combat_env import (
    CombatEnv, SinglePerspectiveEnv, Curriculum, curriculum_for_step,
    OBS_DIM, ACTION_DIM, SQUAD_SIZE, MOVE_DIRS,
    random_opponent, idle_opponent, runner_opponent, make_ga_opponent,
    make_curriculum_opponent_picker, _ga_decide_action,
)
print(f'combat_env loaded. OBS_DIM={OBS_DIM}, ACTION_DIM={ACTION_DIM}')

# Preview the curriculum schedule
print('\nCurriculum schedule (assuming we hit ~30M steps in 7.5h):')
for frac in (0.05, 0.20, 0.40, 0.60, 0.80, 0.95):
    c = curriculum_for_step(int(frac * 30_000_000), 30_000_000)
    stage = 1 if frac < 0.15 else 2 if frac < 0.35 else 3 if frac < 0.55 else 4
    print(f'  t+{frac*100:5.1f}% (stage {stage}): world={c.world_w}x{c.world_h}'
          f' spawn={c.spawn_dist:.0f}u stat={c.p_static:.2f} ga={c.p_ga:.2f}'
          f' self={c.p_self:.2f} vis={c.coef_visibility:.3f}')

## 4. Quick env sanity test

Random actions vs idle in stage 1 should score +500..+1500 (heavy shaping
+ many easy kills). Random actions vs GA-best in stage 4 (no shaping)
should score around 0 — random is a bad strategy.

In [ ]:
import random
import numpy as np

print('Stage 1 (idle opp, random actions):')
c1 = curriculum_for_step(150_000, 1_000_000)
env = CombatEnv(opponent_policy=idle_opponent, curriculum=c1, seed=1)
env.reset(seed=1)
total = 0
for _ in range(env.match_ticks):
    actions = [random.randint(0, ACTION_DIM-1) for _ in range(SQUAD_SIZE)]
    obs, r, done, info = env.step(actions)
    total += sum(r)
    if done: break
print(f'  total_reward={total:.1f}, kills={env.team_kills}')

print('Stage 4 (GA-best opp, random actions):')
c4 = curriculum_for_step(900_000, 1_000_000)
DEFAULT_GA_GENOME = {
    'view_arc': 2.4, 'view_range': 720, 'snap_to_threat': 0.27,
    'patrol_scan_speed': 0.04, 'engage_distance': 280, 'spread': 0.07,
    'fire_cd_frames': 12, 'cover_dmg_threshold': 28, 'peek_duration': 55,
    'hide_duration': 40, 'flee_hp_pct': 0.34, 'flank_chance': 0.6,
}
ga_policy = make_ga_opponent(DEFAULT_GA_GENOME)
env = CombatEnv(opponent_policy=ga_policy, curriculum=c4, seed=2)
env.reset(seed=2)
total = 0
for _ in range(env.match_ticks):
    actions = [random.randint(0, ACTION_DIM-1) for _ in range(SQUAD_SIZE)]
    obs, r, done, info = env.step(actions)
    total += sum(r)
    if done: break
print(f'  total_reward={total:.1f}, kills={env.team_kills}')

## 5. Behavior cloning warm-start (5–8 min)

Roll out **GA-best vs GA-best** for `BC_ROLLOUT_STEPS` ticks, recording
every (observation, action) pair. Then train a small MLP to imitate.
The resulting weights become PPO's initialization.

In [ ]:
import time

print(f'Collecting {BC_ROLLOUT_STEPS:,} (obs, action) pairs from GA-vs-GA...')

# Use a stage-3 curriculum so observation distribution is close to what
# the policy will see in the back half of PPO training (deployment-ish scale).
bc_curriculum = curriculum_for_step(int(0.55 * 30_000_000), 30_000_000)
bc_curriculum.p_static = 0.0
bc_curriculum.p_runner = 0.0
bc_curriculum.p_random = 0.0
bc_curriculum.p_ga = 1.0
bc_curriculum.p_self = 0.0

bc_obs = []
bc_act = []

env = CombatEnv(opponent_policy=ga_policy, curriculum=bc_curriculum, seed=12345)
obs_list = env.reset(seed=12345)
collected = 0
t0 = time.time()
while collected < BC_ROLLOUT_STEPS:
    actions = [_ga_decide_action(env.units[i], env, DEFAULT_GA_GENOME)
               for i in range(SQUAD_SIZE)]
    for k in range(SQUAD_SIZE):
        bc_obs.append(obs_list[k].copy())
        bc_act.append(actions[k])
    obs_list, rewards, done, info = env.step(actions)
    collected += SQUAD_SIZE
    if done:
        obs_list = env.reset()

bc_obs = np.array(bc_obs[:BC_ROLLOUT_STEPS], dtype=np.float32)
bc_act = np.array(bc_act[:BC_ROLLOUT_STEPS], dtype=np.int64)

print(f'Collected {len(bc_obs):,} pairs in {time.time()-t0:.1f}s')
unique, counts = np.unique(bc_act, return_counts=True)
print('  action distribution:')
for u, c in zip(unique, counts):
    print(f'    action {u:2d}: {c:>7,} ({100*c/len(bc_act):5.1f}%)')

In [ ]:
# Supervised BC training
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'BC device: {device}')

NET_HIDDEN = [128, 128]

class PolicyMLP(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=NET_HIDDEN):
        super().__init__()
        layers = []
        in_dim = obs_dim
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.Tanh()]
            in_dim = h
        self.shared = nn.Sequential(*layers)
        self.action_head = nn.Linear(in_dim, act_dim)
    def forward(self, x):
        return self.action_head(self.shared(x))

bc_model = PolicyMLP(OBS_DIM, ACTION_DIM).to(device)
opt = torch.optim.Adam(bc_model.parameters(), lr=1e-3)

bc_obs_t = torch.from_numpy(bc_obs).to(device)
bc_act_t = torch.from_numpy(bc_act).to(device)

t0 = time.time()
bs = 1024
n = len(bc_obs_t)
for epoch in range(BC_EPOCHS):
    perm = torch.randperm(n, device=device)
    losses, correct = [], 0
    for i in range(0, n, bs):
        idx = perm[i:i+bs]
        x = bc_obs_t[idx]
        y = bc_act_t[idx]
        logits = bc_model(x)
        loss = F.cross_entropy(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        correct += (logits.argmax(-1) == y).sum().item()
    if (epoch+1) % max(1, BC_EPOCHS // 10) == 0 or epoch == 0:
        print(f'  BC epoch {epoch+1:>3d}/{BC_EPOCHS}: loss={np.mean(losses):.3f}'
              f'  acc={100*correct/n:.1f}%  ({time.time()-t0:.0f}s)')
print(f'BC training done in {time.time()-t0:.1f}s')

## 6. Build the vectorized PPO env with curriculum hooks

Each parallel worker queries `_global_step.value` (shared mp.Value) at every
reset() — that's how the curriculum advances. The training callback bumps
`_global_step.value` based on **elapsed wall-clock time**, so curriculum
stages get fixed fractions of the time budget regardless of throughput.

In [ ]:
import multiprocessing as mp
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor

# Use a single Manager for all shared state. Manager-backed values are
# picklable across any start_method (fork / forkserver / spawn). On
# Kaggle Python 3.12 the default is forkserver, and raw mp.Value('i', 0)
# can't cross that boundary because it uses mmap internally.
_mp_manager = mp.Manager()
_global_step = _mp_manager.Value('i', 0)
_self_pool_paths = _mp_manager.list()
_ga_genome_global = DEFAULT_GA_GENOME
_worker_policy_cache = {}   # per-worker process cache (each worker has its own copy)

def _load_self_policy(path):
    if path in _worker_policy_cache:
        return _worker_policy_cache[path]
    try:
        from stable_baselines3 import PPO as _PPO
        m = _PPO.load(path, device='cpu')
        def _policy(obs_list, env, _m=m):
            obs_arr = np.stack(obs_list).astype(np.float32)
            actions, _ = _m.predict(obs_arr, deterministic=False)
            return [int(a) for a in actions]
        _worker_policy_cache[path] = _policy
        return _policy
    except Exception:
        return None

def make_opponent_for_curriculum(c):
    pool_callables = []
    if c.p_self > 0 and len(_self_pool_paths) > 0:
        for p in list(_self_pool_paths)[-SELF_POOL_MAX:]:
            if not os.path.exists(p):
                continue
            pol = _load_self_policy(p)
            if pol is not None:
                pool_callables.append(pol)
    current = set(list(_self_pool_paths))
    for stale in [k for k in _worker_policy_cache if k not in current]:
        _worker_policy_cache.pop(stale, None)
    picker = make_curriculum_opponent_picker(
        ga_genome=_ga_genome_global, self_pool=pool_callables)
    return picker(c)

def make_env(rank: int):
    def _init():
        seed = 1000 + rank * 7919
        def provider():
            step = _global_step.value
            return curriculum_for_step(step, TOTAL_STEPS)
        return SinglePerspectiveEnv(
            curriculum_provider=provider,
            opponent_factory=make_opponent_for_curriculum,
            squad_size=SQUAD_SIZE,
            seed=seed,
        )
    return _init

# Smoke test single env
test_env = make_env(0)()
obs0, _ = test_env.reset(seed=42)
print(f'Single env: obs shape={obs0.shape}')
total_r = 0
for _ in range(100):
    a = test_env.action_space.sample()
    o, r, d, t, info = test_env.step(a)
    total_r += r
    if d: break
print(f'  reward over 100 steps: {total_r:.2f}')

print(f'Building SubprocVecEnv with {N_ENVS} workers (start_method=fork)...')
# Force fork on Linux for fastest IPC. CUDA isn't initialized yet (PPO model
# is built later), so fork is safe here.
vec_env = SubprocVecEnv([make_env(i) for i in range(N_ENVS)], start_method='fork')
vec_env = VecMonitor(vec_env)
print('Vec env ready.')

## 7. Build PPO model + transfer BC weights

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback

policy_kwargs = dict(
    net_arch=dict(pi=NET_HIDDEN, vf=NET_HIDDEN),
    activation_fn=nn.Tanh,
)

print('Building PPO model...')
# NOTE: ent_coef is a CONSTANT here (SB3 PPO expects float, not a schedule
# function — unlike learning_rate / clip_range which do accept schedules).
# The training callback updates self.model.ent_coef each step to implement
# the linear decay from ENT_COEF_INIT to ENT_COEF_FINAL.
model = PPO(
    policy='MlpPolicy',
    env=vec_env,
    learning_rate=LR_INIT,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    clip_range=CLIP_RANGE,
    ent_coef=ENT_COEF_INIT,    # callback decays this over time
    policy_kwargs=policy_kwargs,
    verbose=1,
    device=device,
)

# Transfer BC weights into PPO policy
print('Transferring BC weights...')
sb3_pi  = model.policy.mlp_extractor.policy_net
sb3_act = model.policy.action_net
bc_layers = [m for m in bc_model.shared if isinstance(m, nn.Linear)]
sb3_layers = [m for m in sb3_pi if isinstance(m, nn.Linear)]
print(f'  BC linear layers: {len(bc_layers)}, SB3 linear layers: {len(sb3_layers)}')

if len(bc_layers) == len(sb3_layers):
    for bc_l, sb_l in zip(bc_layers, sb3_layers):
        if bc_l.weight.shape == sb_l.weight.shape:
            sb_l.weight.data.copy_(bc_l.weight.data)
            sb_l.bias.data.copy_(bc_l.bias.data)
    if bc_model.action_head.weight.shape == sb3_act.weight.shape:
        sb3_act.weight.data.copy_(bc_model.action_head.weight.data)
        sb3_act.bias.data.copy_(bc_model.action_head.bias.data)
        print('  Action head transferred OK.')
    else:
        print(f'  WARN: action head shape mismatch.')
else:
    print('  WARN: layer count mismatch — random init.')

# Free BC dataset to reclaim GPU RAM
del bc_obs, bc_act, bc_obs_t, bc_act_t
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

n_params = sum(p.numel() for p in model.policy.parameters())
print(f'PPO model ready. Parameters: {n_params:,}')

## 8. Training callback: time budget + curriculum + self-play snapshots

Three things this callback does, every SB3 step:
1. **Time budget**: stops `model.learn()` when `TIME_LIMIT_HOURS` elapsed.
2. **Curriculum advance**: maps elapsed-time-fraction onto the step ceiling
   so curriculum stages take fixed fractions of wall-clock time.
3. **Self-play snapshots**: every `FROZEN_OPP_EVERY` steps, save the
   current policy and add it to the opponent pool.

In [ ]:
class CurriculumSelfPlayCallback(BaseCallback):
    def __init__(self, output_dir: str, deadline_ts: float, verbose=0):
        super().__init__(verbose)
        self.output_dir = output_dir
        self.deadline_ts = deadline_ts
        self.last_freeze = 0
        self.last_checkpoint = 0
        self.last_log_t = 0.0
        self.t_start = time.time()
        self.stopped_for_time = False

    def _on_step(self) -> bool:
        n = self.num_timesteps
        elapsed = time.time() - self.t_start
        deadline_elapsed = self.deadline_ts - self.t_start
        time_progress = max(0.0, min(1.0, elapsed / max(1.0, deadline_elapsed)))
        virtual_step = int(time_progress * TOTAL_STEPS)
        _global_step.value = virtual_step

        # Linear entropy-coefficient decay (SB3 doesn't accept ent_coef
        # schedules natively, so we patch it from here)
        self.model.ent_coef = ENT_COEF_FINAL + (1 - time_progress) * (ENT_COEF_INIT - ENT_COEF_FINAL)

        # Time-based stop
        if time.time() >= self.deadline_ts:
            print(f'\n[time] deadline at step {n:,} '
                  f'(elapsed {elapsed/3600:.2f}h / {deadline_elapsed/3600:.2f}h budget) — stopping.')
            self.stopped_for_time = True
            return False

        # Self-play snapshot every FROZEN_OPP_EVERY virtual steps
        if virtual_step - self.last_freeze >= FROZEN_OPP_EVERY:
            self.last_freeze = virtual_step
            path = os.path.join(self.output_dir, f'self_snap_{n}.zip')
            try:
                self.model.save(path)
                _self_pool_paths.append(path)
                while len(_self_pool_paths) > SELF_POOL_MAX:
                    old = _self_pool_paths.pop(0)
                    try:
                        if os.path.exists(old): os.remove(old)
                    except OSError: pass
                if self.verbose:
                    print(f'  [selfplay] snap @ step {n:,}: pool={len(_self_pool_paths)}')
            except Exception as e:
                print(f'  [selfplay] snap failed: {e}')

        # Real-step checkpoints (difficulty tiers)
        if n - self.last_checkpoint >= CHECKPOINT_EVERY:
            self.last_checkpoint = n
            path = os.path.join(self.output_dir, f'ppo_combat_{n}.zip')
            try:
                self.model.save(path)
                if self.verbose:
                    print(f'  [ckpt] @ step {n:,} -> {path}')
            except Exception as e:
                print(f'  [ckpt] save failed: {e}')

        # Status log every ~2 minutes wall-clock
        if elapsed - self.last_log_t >= 120:
            self.last_log_t = elapsed
            c = curriculum_for_step(virtual_step, TOTAL_STEPS)
            stage = 1 if time_progress < 0.15 else 2 if time_progress < 0.35 else 3 if time_progress < 0.55 else 4
            sps = n / max(1.0, elapsed)
            eta_h = (deadline_elapsed - elapsed) / 3600
            print(f'[t+{elapsed/60:>6.1f}m  s{stage}] step {n:>10,} '
                  f'({sps:>5.0f}/s, eta {eta_h:>4.2f}h) '
                  f'world={c.world_w}x{c.world_h} spawn={c.spawn_dist:.0f} '
                  f'ga={c.p_ga:.2f} self={c.p_self:.2f} pool={len(_self_pool_paths)} '
                  f'vis={c.coef_visibility:.3f}')
        return True

print('Callback class defined.')

## 9. Train (long-running cell — go to bed)

The progress bar is disabled to keep the log readable. Instead, the
callback prints a status line every ~2 minutes:

```
[t+ 12.3m s1] step  4,829,184 ( 873/s, eta 7.31h) world=600x600 spawn=240 ga=0.07 ...
```

Look for `s1 → s2 → s3 → s4` transitions and `ep_rew_mean` in SB3's own
periodic dump. If `ep_rew_mean` rises monotonically you're winning.

In [ ]:
deadline_ts = time.time() + TIME_LIMIT_HOURS * 3600
print(f'>>> Training until {time.strftime("%H:%M:%S", time.localtime(deadline_ts))} '
      f'({TIME_LIMIT_HOURS}h from now)')
print(f'>>> Step ceiling: {TOTAL_STEPS:,}')

t0 = time.time()
cb = CurriculumSelfPlayCallback(
    output_dir=OUTPUT_DIR, deadline_ts=deadline_ts, verbose=1)
model.learn(total_timesteps=TOTAL_STEPS, callback=cb, progress_bar=False)

elapsed = time.time() - t0
print(f'\n>>> Training done. Wall: {elapsed/3600:.2f}h')
print(f'>>> Real steps: {model.num_timesteps:,}')
print(f'>>> Stopped because: {"time-limit" if cb.stopped_for_time else "step-ceiling"}')

final_path = os.path.join(OUTPUT_DIR, 'ppo_combat_final.zip')
model.save(final_path)
print(f'>>> Final model: {final_path}')

## 10. Export ONNX (for the JS game's onnxruntime-web)

In [ ]:
class OnnxablePolicy(nn.Module):
    def __init__(self, sb3_policy):
        super().__init__()
        self.extractor = sb3_policy.mlp_extractor
        self.action_net = sb3_policy.action_net
    def forward(self, obs):
        latent_pi = self.extractor.policy_net(obs)
        logits = self.action_net(latent_pi)
        return torch.softmax(logits, dim=-1)

# Move to CPU for export
model.policy.to('cpu')
onnxable = OnnxablePolicy(model.policy).eval()

dummy = torch.zeros(1, OBS_DIM, dtype=torch.float32)
onnx_path = os.path.join(OUTPUT_DIR, 'model.onnx')
torch.onnx.export(
    onnxable, dummy, onnx_path,
    input_names=['obs'], output_names=['action_probs'],
    dynamic_axes={'obs': {0: 'batch'}, 'action_probs': {0: 'batch'}},
    opset_version=17,
    dynamo=False,
)
print(f'ONNX -> {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)')

# Verify
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
test_in = np.random.randn(1, OBS_DIM).astype(np.float32)
out = sess.run(None, {'obs': test_in})
print(f'  inference: {test_in.shape} -> {out[0].shape}')
print(f'  sample probs: {out[0][0][:6].round(3)}... (sum={out[0][0].sum():.3f})')

if torch.cuda.is_available():
    model.policy.to('cuda')

## 11. Sanity check: NN vs GA-best at deployment scale (20 matches)

In [ ]:
print('Running 20 NN-vs-GA-best matches at deployment scale (1200x1200, 700u spawn)...')
deploy_curriculum = curriculum_for_step(TOTAL_STEPS, TOTAL_STEPS)  # final stage

wins = losses = draws = 0
ep_rewards = []
kill_diffs = []

for match in range(20):
    env = CombatEnv(opponent_policy=ga_policy,
                    curriculum=deploy_curriculum,
                    seed=20000 + match)
    obs_list = env.reset(seed=20000 + match)
    total_reward = 0.0
    for _ in range(env.match_ticks):
        actions = []
        for k in range(SQUAD_SIZE):
            obs = obs_list[k].astype(np.float32)
            action, _ = model.predict(obs, deterministic=True)
            actions.append(int(action))
        obs_list, rewards, done, info = env.step(actions)
        total_reward += sum(rewards) / SQUAD_SIZE
        if done: break
    ep_rewards.append(total_reward)
    kill_diffs.append(env.team_kills[0] - env.team_kills[1])
    if   env.team_kills[0] > env.team_kills[1]: wins   += 1
    elif env.team_kills[1] > env.team_kills[0]: losses += 1
    else:                                       draws  += 1
    if (match+1) % 5 == 0:
        print(f'  match {match+1:>2}/20: kills NN={env.team_kills[0]} GA={env.team_kills[1]}'
              f'  W/L/D = {wins}/{losses}/{draws}')

print(f'\nFINAL:  {wins} W / {losses} L / {draws} D    win-rate {wins/20*100:.0f}%')
print(f'Mean episode reward: {np.mean(ep_rewards):+.1f}')
print(f'Mean kill differential (NN - GA): {np.mean(kill_diffs):+.2f}')

if wins >= 14: print('[EXCELLENT] NN clearly beats GA-best.')
elif wins >= 10: print('[OK] NN comparable or better than GA-best.')
elif wins >= 6:  print('[WEAK] Mixed — try more time or check stage 4 ep_rew_mean.')
else:            print('[BAD] Try doubling BC_ROLLOUT_STEPS or raising TIME_LIMIT_HOURS.')

## 12. Output files (Kaggle right-side panel → "Output")

- **`/kaggle/working/ppo_ckpt/model.onnx`** ← drop this into the AshGrid repo
  at `ai_arena/onnx/model.onnx`. The JS-side wiring already exists on the
  `claude/ai-ppo-nn` branch — just replace the file and reload the page.
- **`ppo_combat_<step>.zip`** — checkpoints every 2M real steps. To produce
  difficulty tiers, re-run cells 7+10 after `model = PPO.load("path/to/<step>.zip")`,
  rename outputs to `model_easy.onnx` / `model_medium.onnx` / `model_hard.onnx`.
- **`ppo_combat_final.zip`** — full SB3 model, can resume training from here.

Right-click → Download from the Output panel.